# Session6 — Public 0.326

Воспроизведение отправленного прогноза из сохранённых весов на CPU. **Public 0.326 сообщён пользователем; цель Public < 0.322 пока не достигнута.**

Локальная WAPE + |Rbias| = **0.319190199953**, средняя дневная комбинация = **0.336038661802**. Локальный период многократно использовался в разработке, поэтому эти оценки не заменяют Public.

Все необходимые исходные parquet, веса, признаки и эталонный CSV находятся в этой папке. Выполняйте ячейки сверху вниз в существующем Python-окружении с зависимостями из `bundle/requirements-reproduce.txt`. Полное обучение не запускается.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import sys

import numpy as np
import pandas as pd

locations = [Path.cwd(), Path.cwd()/'best', Path.cwd().parent/'best',
             Path.cwd()/'session6']
ROOT = next((p.resolve() for p in locations
             if (p/'bundle/manifest.json').is_file() and (p/'data/train_solo_track.parquet').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Укажите папку best в ROOT: рядом должны быть bundle/ и data/.')
BUNDLE, DATA = ROOT/'bundle', ROOT/'data'
manifest = json.loads((BUNDLE/'manifest.json').read_text())
print('Папка:', ROOT)
print('Python:', sys.executable)
print('Закрытый тест:', manifest['public_new'])
print('Локальная комбинация:', manifest['local_score'])

## Проверка входных файлов

Проверяем содержимое комплекта по SHA-256. Исходные parquet не изменяются; кэш признаков и новый CSV записываются отдельно.

In [ ]:
input_hashes = json.loads((ROOT/'input_hashes.json').read_text())
for relative, expected_hash in input_hashes.items():
    path = ROOT/relative
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open('rb') as stream:
        actual_hash = hashlib.file_digest(stream, 'sha256').hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f'Изменён входной файл: {relative}')
print(f'Проверены {len(input_hashes)} входных файлов')

## Рецепт и временная схема

Восемь LightGBM-моделей из исторической итерации 86 задают распределение по маршрутам. Для каждого горизонта их прогноз нормируется до суммы исходного 0328. Вес LightGBM — **0.49972590220122615**, вес 0328 — **0.5002740977987738**. Суммарный прогноз каждого горизонта сохраняется.

Базовые модели обучены по меткам до 4 октября 2025 года; early stopping использовал 4–10 октября. Вес смеси выбран по 11–17 октября через scipy.optimize, для 0328 применялся временной crossfit. Проверка — 18–31 октября, 14 стартов в 10:30, 1000 маршрутов × 8 горизонтов по 30 минут. Сохранённые OOS-прогнозы ниже позволяют пересчитать метрики; это не новое обучение и не новая независимая проверка.

In [ ]:
validation = pd.read_parquet(ROOT/'validation_predictions.parquet')
assert len(validation) == 112000
assert not validation.duplicated(['cutoff', 'horizon', 'route_id']).any()

def evaluate(frame):
    y = frame.y.to_numpy(dtype=np.float64)
    p = frame.session_prediction.to_numpy(dtype=np.float64)
    if y.sum() <= 0:
        raise ValueError('Нулевой знаменатель метрики')
    assert np.isfinite(p).all() and (p >= 0).all()
    error = p-y
    wape = float(np.abs(error).sum()/y.sum())
    rbias = float(error.sum()/y.sum())
    return {'WAPE': wape, 'Rbias': rbias, 'WAPE + |Rbias|': wape+abs(rbias)}

pooled = evaluate(validation)
daily = pd.DataFrame([{'Дата старта': str(day), **evaluate(part)}
                      for day, part in validation.groupby('cutoff')])
assert np.isclose(pooled['WAPE + |Rbias|'], manifest['local_score'], rtol=0, atol=1e-12)
display(pd.DataFrame([pooled], index=['Все 112000 строк']))
print('Средняя дневная комбинация:', daily['WAPE + |Rbias|'].mean())
print('Отрицательный Rbias означает занижение; в итоговой метрике берётся модуль.')
display(daily)
display(pd.DataFrame([{'Горизонт, шаги по 30 минут': h, **evaluate(part)}
                      for h, part in validation.groupby('horizon')]))

## Прогноз на конкурсную дату

История заканчивается 1 ноября 2025 года в 10:30. Требуются восемь прогнозов на 11:00–14:30 для каждого маршрута. `target_1h` — перекрывающееся часовое окно; сумма восьми меток не трактуется как уникальный объём за четыре часа.

Следующая ячейка строит признаки из raw train и применяет сохранённые веса. Ей не нужны файлы исходного проекта, доступ к серверу или GPU.

In [ ]:
spec = importlib.util.spec_from_file_location('session6_runtime', BUNDLE/'predict_session.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
submission = runtime.predict(BUNDLE, DATA, ROOT/'.cache')
submission.head()

## Проверка и сохранение CSV

Сверяем все 8000 строк с отправленным CSV session6. Разные версии NumPy/pandas могут отличаться округлением последних битов, поэтому сравнение чисел использует строгий допуск. Отправленный исходный файл сохранён отдельно как `submission_session6.csv`.

In [ ]:
expected = pd.read_csv(ROOT/'submission_session6.csv')
test = pd.read_parquet(DATA/'test_solo_track.parquet', columns=['id', 'timestamp', 'route_id'])
assert list(submission.columns) == ['id', 'y_pred']
assert len(submission) == 8000 and submission.id.is_unique
assert np.array_equal(submission.id, test.id) and np.array_equal(submission.id, expected.id)
assert np.isfinite(submission.y_pred).all() and submission.y_pred.ge(0).all()
np.testing.assert_allclose(submission.y_pred, expected.y_pred, rtol=1e-12, atol=1e-8)

old = pd.read_csv(BUNDLE/'reference_0328.csv').rename(columns={'y_pred': 'old_prediction'})
totals = test.merge(submission, on='id', validate='one_to_one').merge(old, on='id', validate='one_to_one')
totals = totals.groupby('timestamp')[['y_pred', 'old_prediction']].sum()
np.testing.assert_allclose(totals.y_pred, totals.old_prediction, rtol=1e-12, atol=1e-6)

output = ROOT/'outputs/submission_session6_reproduced.csv'
output.parent.mkdir(exist_ok=True)
submission.to_csv(output, index=False)
print('Сохранено:', output)
print('Максимальное отклонение от отправленного CSV:', float(np.abs(submission.y_pred-expected.y_pred).max()))
print('Все 8000 прогнозов проверены; суммы по горизонтам сохранены.')